# Taller S3 · CONECTA S.A., Grupo #

**Curso:** Visualización de Datos y Gerencia de la Información (UNAULA)

**Integrantes:** Maria Camila Grajales Perez, Stiware Alexander Quintero Aguirre, Juan Pablo Ramirez Gomez

**Grupo:** _#1_

## Propósito

Este taller integra las actividades de las sesiones S3 y S4 en un solo entregable. En la **Parte A (S3)** se diagnostican y corrigen los problemas de calidad del dataset de su grupo, aplicando los métodos formales vistos en clase (VIF, missingno, tres métodos de outliers, comparación de imputación). En la **Parte B (S4)** se toma un hallazgo sobre esos datos ya limpios, se verifica que no sea el efecto de una variable de confusión no controlada, y se comunica mediante un SCQA y un gráfico efectivo.

No existe una única forma correcta de limpiar estos datos ni un único hallazgo válido. Lo que se evalúa es que cada decisión esté justificada con un criterio explícito y, donde aplique, con el método formal correspondiente, no que coincida con una respuesta predefinida por el docente.

## Cómo usar este notebook

1. Dupliquen este archivo y renómbrenlo con el número de su grupo (por ejemplo `v0.1_s3_dataviz_grupo3.ipynb`).
2. En la celda de configuración, cambien `GRUPO` por su número (1 a 5) para cargar el archivo correcto.
3. Completen cada actividad en las celdas marcadas con `# TODO`. Las celdas de conclusión siguen el patrón de clase: bullets basados en los números y gráficos efectivamente obtenidos.
4. **Entregables:** este mismo notebook con las dos partes resueltas, más el archivo `dataset_limpio.csv` generado al final de la Parte A.

## 0. Configuración

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.ensemble import IsolationForest
from sklearn.impute import KNNImputer
import os

pd.set_option("display.float_format", lambda x: f"{x:.2f}")

INDIGO = "#4F46E5"
sns.set_theme(style="whitegrid", font_scale=1.0)

GRUPO = 1  # cambien este número por el de su grupo (1 a 5)

df = pd.read_csv(os.path.join("data", f"conecta_sa_grupo{GRUPO}.csv"))
df.head()

### Una función auxiliar que van a necesitar varias veces

`region` y `plan_type` presentan variantes de escritura (mayúsculas, tildes faltantes, espacios adicionales). A continuación, una función auxiliar para normalizarlas, con la misma lógica empleada en el taller de S2:

In [ ]:
def normalizar_texto(valor):
    texto = str(valor).strip().upper()
    for con_tilde, sin_tilde in [("Á", "A"), ("É", "E"), ("Í", "I"), ("Ó", "O"), ("Ú", "U")]:
        texto = texto.replace(con_tilde, sin_tilde)
    return texto.title()

# Parte A · Limpieza de datos (S3)

Diagnostiquen y corrijan los problemas de calidad del dataset de su grupo aplicando los métodos formales vistos en clase, en lugar de decisiones basadas solo en inspección visual. El resultado de esta parte es `dataset_limpio.csv`, que también se usa como insumo de la Parte B.

## A1. Diagnóstico cuantificado

**Propósito:** para cada columna del dataset, reporten el porcentaje de nulos. Para las columnas de texto (`region`, `plan_type`), la lista de valores únicos que encuentren con `.unique()` o `.value_counts()`.

In [ ]:
# TODO: porcentaje de nulos por columna, ordenado de mayor a menor
df.isnull().mean().sort_values(ascending=False) * 100

# TODO: valores únicos de las columnas de texto
# df["region"].value_counts()
# df["plan_type"].value_counts()

**Su conclusión:** _(qué columnas tienen nulos y en qué porcentaje, qué variantes de texto encontraron en `region` y `plan_type`)_

## A2. Multicolinealidad con VIF

**Propósito:** calculen el Variance Inflation Factor (statsmodels) para las variables numéricas de su dataset (`tenure_months`, `monthly_revenue_cop`, `data_usage_gb`, `complaints_month`). ¿Alguna supera el umbral severo de 10? Si no hay colinealidad severa, declárenlo explícitamente, no lo omitan.

In [ ]:
# TODO: VIF para tenure_months, monthly_revenue_cop, data_usage_gb, complaints_month
# mismo procedimiento visto en clase: sm.add_constant, variance_inflation_factor por columna

**Su conclusión:** _(VIF de cada variable, si alguna supera 10 o 5; si no hay colinealidad severa, declárenlo explícitamente)_

## A3. Patrón de ausencia con missingno

**Propósito:** usen `msno.matrix` sobre el dataset completo. Como `nps_score` tiene muchos más nulos que cualquier otra columna (~80%), ¿el patrón visual sugiere que la ausencia se concentra en algún tramo del archivo, o está dispersa?

In [ ]:
# TODO: msno.matrix(df)

**Su conclusión:** _(qué muestra la matriz sobre el patrón de ausencia de `nps_score` y de las demás columnas)_

## A4. Coordenadas

**Propósito:** revisen `latitud` y `longitud`. Colombia está aproximadamente entre 0° y 13° de latitud norte, y entre -66° y -80° de longitud. Identifiquen nulos, valores en (0, 0), y valores fuera de rango. Decidan qué hacer con cada tipo de problema.

In [ ]:
# TODO: nulos en latitud/longitud
# TODO: valores en (0, 0), "null island"
# TODO: valores fuera de rango de Colombia (latitud 0-13, longitud -80 a -66)

**Su conclusión:** _(cuántos casos de cada tipo encontraron, y qué decidieron hacer con cada uno: eliminar, marcar, imputar)_

## A5. Texto inconsistente

**Propósito:** normalicen `region` y `plan_type` a un conjunto único de valores. Documenten qué variantes encontraron antes de unificarlas.

In [ ]:
# TODO: apliquen normalizar_texto() a region y plan_type
# antes de sobreescribir, comparen value_counts() de antes vs. después para documentar las variantes

**Su conclusión:** _(qué variantes encontraron, y el conteo antes/después de normalizar)_

## A6. Duplicados

**Propósito:** verifiquen si hay filas exactamente duplicadas. Antes de eliminarlas, confirmen que realmente son duplicados de carga y no dos eventos legítimos del mismo usuario en el mismo mes.

In [ ]:
# TODO: df.duplicated().sum()
# TODO: revisen si los duplicados coinciden en user_id + mes, o son coincidencias parciales en otras columnas

**Su conclusión:** _(cuántos duplicados encontraron, y por qué concluyeron que son duplicados de carga y no eventos legítimos, o viceversa)_

## A7. El caso de `nps_score`: ¿por qué falta?

**Propósito:** calculen el % de nulos en `nps_score`, y compárenlo entre usuarios con `churned=1` y `churned=0`. En clase vieron tres tipos de ausencia (MCAR, MAR, MNAR): usen esa comparación, junto con lo que ya saben del negocio de CONECTA, para argumentar cuál de los tres aplica aquí y por qué, no solo para reportar el número. Con esa conclusión, justifiquen por escrito qué van a hacer con la columna: ¿la imputan, dejan el nulo, o la excluyen de ciertos análisis?

In [ ]:
# TODO: % de nulos en nps_score, total y por grupo de churned (0 vs 1)

**Su conclusión:** _(el % de nulos por grupo, si es MCAR/MAR/MNAR según ese indicio, y qué decidieron hacer con la columna)_

## A8. Outliers: tres métodos, no uno solo

**Propósito:** apliquen a `monthly_revenue_cop` los tres métodos vistos en clase: Tukey (Q1-1.5×IQR a Q3+1.5×IQR), Modified Z-score con MAD (Iglewicz & Hoaglin, 1993, umbral 3.5), e Isolation Forest (multivariado, sobre `monthly_revenue_cop`, `data_usage_gb`, `complaints_month`). Comparen los tres conjuntos de outliers: ¿coinciden en gran medida, o cada método señala casos distintos? Relacionen la diferencia con lo que ya observaron sobre estas variables en el correlograma del taller de S2.

In [ ]:
# TODO: Tukey IQR sobre monthly_revenue_cop
# TODO: Modified Z-score (MAD) sobre monthly_revenue_cop
# TODO: Isolation Forest sobre monthly_revenue_cop, data_usage_gb, complaints_month

**Su conclusión:** _(cuántos outliers detectó cada método, y en qué medida coinciden o difieren entre sí)_

## A9. Imputación comparada, si aplica

**Propósito:** si su grupo decide imputar algún valor (no necesariamente coordenadas), comparen al menos dos métodos (por ejemplo mediana por grupo vs. KNN) con un gráfico de densidades superpuestas antes de aplicar uno. Si el dataset no requiere imputación en ninguna columna numérica, documenten por qué no es necesaria, en vez de omitir el punto.

In [ ]:
# TODO: si aplica, comparen al menos 2 métodos de imputación con un gráfico de densidades superpuestas
# (sns.kdeplot de la variable original sin nulos contra cada método candidato)

**Su conclusión:** _(qué columna imputaron y con qué método, o por qué no fue necesario imputar nada)_

## A10. Documentación de decisiones

**Propósito:** por cada corrección aplicada, una línea: qué se encontró, qué método formal se usó (si aplica), qué se decidió, y por qué.

| problema encontrado | método formal usado | decisión | por qué |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |

In [ ]:
# TODO: apliquen aquí todas las correcciones que decidieron arriba, sobre una copia del dataframe
df_limpio = df.copy()

# ...

df_limpio.to_csv(os.path.join("data", "dataset_limpio.csv"), index=False)
print("Nulos restantes:", df_limpio.isnull().sum().sum())
df_limpio.head()

## Entregable Parte A

`dataset_limpio.csv` (ya guardado arriba) más la tabla de decisiones del punto A10 completa.

# Parte B · SCQA y diseño efectivo (S4)

Tomen un hallazgo sobre `dataset_limpio.csv` (generado en la Parte A), verifiquen que se sostenga al considerar otra variable relevante, y comuníquenlo mediante un SCQA y un gráfico efectivo. El centro de esta parte es el SCQA y una primera versión del gráfico que lo sostenga: no tiene que ser la versión final, la van a seguir refinando en sesiones posteriores del curso. El heatmap de `region` x `tier_localidad` y el choropleth construidos en el taller de S2 ya sugieren que ambas dimensiones no son independientes entre sí; tengan esto presente al elegir qué variable considerar.

In [ ]:
df = pd.read_csv(os.path.join("data", "dataset_limpio.csv"))
df.head()

## B1. Hallazgo y verificación

**Propósito:** identifiquen un hallazgo relevante para CONECTA (por ejemplo, una diferencia en `churned` o en `monthly_revenue_cop` asociada a alguna variable). Verifíquenlo considerando al menos una variable adicional (`region`, `tier_localidad`, `mes`, `plan_type`), con el método que corresponda a su pregunta: regresión, comparación de tasas o promedios, tabla cruzada. No todo hallazgo requiere un modelo. Reporten si el hallazgo se sostiene, se debilita, o desaparece al considerar esa variable.

In [ ]:
# TODO: identifiquen un hallazgo (comparación, correlación o modelo simple, según corresponda)
# TODO: verifíquenlo considerando al menos una variable adicional relevante

**Su conclusión:** _(cuál es el hallazgo, y si se sostiene o cambia al considerar la variable adicional)_

## B2. Escribir el SCQA

**Propósito:** con el hallazgo ya verificado, escriban el SCQA antes de tocar el código del gráfico.

**Situación:** _(contexto que cualquier persona en CONECTA acepta como punto de partida)_

**Complicación:** _(qué cambió o qué tensión rompe esa situación)_

**Pregunta:** _(la pregunta implícita que deja la complicación)_

**Respuesta:** _(la conclusión que su gráfico va a sostener, basada en el hallazgo verificado de B1)_

## B3. Primera versión del gráfico efectivo

**Propósito:** construyan una primera versión del gráfico que sostenga la Respuesta del SCQA, aplicando al menos cuatro principios de diseño: tinta proporcional, manejo de superposición si aplica, color con propósito (no arcoíris), codificación redundante si ayuda a accesibilidad, eliminación de desorden, balance dato-contexto. El título debe ser, literalmente, la Respuesta del SCQA. Es un primer intento (MVP), no la versión final: la van a seguir refinando en sesiones posteriores del curso.

In [ ]:
# TODO: primera versión del gráfico, con el título = la Respuesta del SCQA
# marquen en un comentario cuáles principios aplicaron

**Su conclusión:** _(qué principios aplicaron, y qué le falta pulir en próximas versiones)_

## Entregable Parte B

El hallazgo verificado, el SCQA escrito, y la primera versión del gráfico efectivo (código incluido), con los principios de diseño aplicados documentados.

## Entregable completo de este taller

Este mismo notebook (`v0.1_s3_dataviz_grupoN.ipynb`, con su número de grupo), con la Parte A y la Parte B resueltas, más el archivo `dataset_limpio.csv` generado en A10. Antes de entregar, revisen que cada conclusión esté basada en los números y gráficos efectivamente obtenidos, y no en lo que esperaban encontrar.